# Authenticity Analysis

Notebook for dataset experiments, training runs, validation studies, and confusion-matrix evaluation. Model definition, inference, explainability, and video runtime logic live in the backend package.

## Scope

Keep in this notebook:
- dataset preparation and splits
- training experiments
- validation metrics and confusion matrix studies

Move out of this notebook:
- model definition
- inference pipeline
- Grad-CAM and ViT rollout generation
- video frame prediction logic

In [ ]:
from pathlib import Path
import sys

import torch
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

repo_root = Path.cwd().parents[0] if Path.cwd().name == 'training' else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from backend.app.ml.model_def import HybridCNNViT

DATASET_ROOT = repo_root / 'dataset'
INPUT_SIZE = 224
BATCH_SIZE = 16


def notebook_context() -> dict[str, object]:
    return {
        'torch_version': torch.__version__,
        'data_loader_type': DataLoader,
        'subset_type': Subset,
        'dataset_module': datasets,
        'transform_module': transforms,
        'model_class': HybridCNNViT,
        'dataset_root': DATASET_ROOT,
        'input_size': INPUT_SIZE,
        'batch_size': BATCH_SIZE,
    }


notebook_context()

## Dataset Preparation

This section records the dataset arrangement and the train/validation split used during experimentation. Keep any one-off data wrangling here so it remains reproducible.

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print(f'Dataset root: {DATASET_ROOT}')

## Training Experiments

The model class is imported from the backend package, but the notebook remains the place to compare training settings, record losses, and note any experiment-specific changes.

In [ ]:
model = HybridCNNViT(num_classes=2, input_size=INPUT_SIZE)
train_loader = DataLoader(Subset([], []), batch_size=BATCH_SIZE)
val_loader = DataLoader(Subset([], []), batch_size=BATCH_SIZE)
print(model)

## Validation and Confusion Matrix

Keep confusion-matrix studies, precision, recall, F1, and any error analysis here. This is the right place for performance interpretation after each experiment.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

all_labels = [0, 1, 0, 1]
all_preds = [0, 1, 1, 1]

cm = confusion_matrix(all_labels, all_preds)
print(cm)
print(classification_report(all_labels, all_preds, target_names=['Fake', 'Real']))

## Export Artifacts

Once training stabilizes, export the checkpoint and metadata for backend consumption. The backend should read these artifact files rather than re-running notebook logic.

In [ ]:
from export_artifacts import main as export_artifacts

export_artifacts()
print('Artifacts exported to the artifacts directory.')